### DoubleDQN vs Random

In [1]:
import torch
from collections import Counter
from domain.configs import MAX_STEPS_PER_EPISODE
from environment.absolute_perspective_version.grenight_environment import GrenightEnvironment
from agents.double_dqn_vs_random.agent import Agent

In [2]:
def play_game(env_arg: GrenightEnvironment,
              agent_arg: Agent) -> tuple[str, dict, int]:

    state = env_arg.reset()
    done = False
    move_count = 0
    acting_player_is_white = True
    reward = 0.0
    info = None

    while not done and move_count < MAX_STEPS_PER_EPISODE:
        acting_player_is_white = env_arg.is_white_on_turn
        if acting_player_is_white:
            action = agent_arg.select_action(state, env_arg.action_mask(), 0.0)
        else:
            action = env_arg.sample()
        state, reward, done, info = env_arg.step(action)
        move_count += 1

    if not done:
        return "truncated", info, move_count
    if reward == 0.0:
        return "draw", info, move_count

    winner_is_white = acting_player_is_white if reward == 1.0 else not acting_player_is_white

    return ("white_win", info, move_count) if winner_is_white else ("black_win", info, move_count)

In [5]:
device = "cpu"

for i in range(1, 6):
    env = GrenightEnvironment()
    outcomes_counter = Counter()
    draw_reasons_counter = Counter()

    agent = Agent(
        num_planes=env.state_encoder.NUM_PLANES,
        rows=5,
        columns=4,
        num_actions=env.action_encoder.NUM_ACTIONS,
        device=device
    )

    checkpoint = torch.load(f"../double_dqn_vs_random/checkpoints/ep{i * 5_000}.pt", map_location=device, weights_only=False)

    agent.policy_net.load_state_dict(checkpoint["policy_state_dict"])
    agent.target_net.load_state_dict(checkpoint["target_state_dict"])
    agent.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    for _ in range(1_000):
        outcome, game_info, move_count = play_game(env, agent)
        outcomes_counter[outcome] += 1
        if game_info["draw_reason"] is not None:
            draw_reasons_counter[game_info["draw_reason"]] += 1

    print(f"STATS OUT FROM: {1_000} GAMES AT: {i * 5_000} CHECKPOINT\n"
          f"Outcomes: {outcomes_counter}\n"
          f"Draw reasons: {draw_reasons_counter}\n")

STATS OUT FROM: 1000 GAMES AT: 5000 CHECKPOINT
Outcomes: Counter({'draw': 677, 'white_win': 312, 'black_win': 11})
Draw reasons: Counter({'threefold_repetition': 314, 'stalemate': 305, 'insufficient_material': 46, 'max_steps_without_progress': 12})

STATS OUT FROM: 1000 GAMES AT: 10000 CHECKPOINT
Outcomes: Counter({'white_win': 714, 'draw': 278, 'black_win': 8})
Draw reasons: Counter({'stalemate': 136, 'threefold_repetition': 134, 'insufficient_material': 6, 'max_steps_without_progress': 2})

STATS OUT FROM: 1000 GAMES AT: 15000 CHECKPOINT
Outcomes: Counter({'white_win': 895, 'draw': 92, 'black_win': 13})
Draw reasons: Counter({'threefold_repetition': 41, 'stalemate': 39, 'insufficient_material': 10, 'max_steps_without_progress': 2})

STATS OUT FROM: 1000 GAMES AT: 20000 CHECKPOINT
Outcomes: Counter({'white_win': 943, 'draw': 55, 'black_win': 2})
Draw reasons: Counter({'stalemate': 25, 'threefold_repetition': 23, 'insufficient_material': 7})

STATS OUT FROM: 1000 GAMES AT: 25000 CHECKP